# SASRec inference-only test

Load a saved SASRec checkpoint and run validation/test evaluation plus top-k inference without training.
The model hyperparameters must match the checkpoint that was saved during training.

In [1]:
from pathlib import Path
import sys
from types import SimpleNamespace

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:\Users\hyewoo choi\Documents\99. 대학원\0. 논문\git\time-aware-behavior-prediction


In [2]:
import torch

from src.sasrec_utils import evaluate, load_sasrec_dataset, recommend_topk
from src.train_sasrec import init_model, set_seed

print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())

torch: 2.11.0+cpu
cuda_available: False


In [3]:
args = SimpleNamespace(
    interactions_path=str(PROJECT_ROOT / 'sample_data' / 'bpi2012_500_cases' / 'sasrec_interactions.txt'),
    checkpoint_path=str(PROJECT_ROOT / 'outputs' / 'sasrec_notebook_500' / 'sasrec_notebook_last.pth'),
    maxlen=50,
    hidden_units=16,
    num_blocks=1,
    num_heads=1,
    dropout_rate=0.2,
    device='cpu',
    norm_first=False,
    eval_users=100,
    num_negative_samples=20,
    topk=10,
    seed=42,
)

checkpoint_path = Path(args.checkpoint_path)
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

checkpoint_path

WindowsPath('C:/Users/hyewoo choi/Documents/99. 대학원/0. 논문/git/time-aware-behavior-prediction/outputs/sasrec_notebook_500/sasrec_notebook_last.pth')

In [4]:
set_seed(args.seed)
dataset = load_sasrec_dataset(args.interactions_path)
user_train, user_valid, user_test, user_num, item_num = dataset

print('users:', user_num)
print('items:', item_num)
print('avg_train_len:', sum(len(v) for v in user_train.values()) / len(user_train))

users: 500
items: 23
avg_train_len: 12.14


In [5]:
model = init_model(user_num, item_num, args)
state_dict = torch.load(checkpoint_path, map_location=args.device)
model.load_state_dict(state_dict)
model.eval()

print('loaded:', checkpoint_path)

loaded: C:\Users\hyewoo choi\Documents\99. 대학원\0. 논문\git\time-aware-behavior-prediction\outputs\sasrec_notebook_500\sasrec_notebook_last.pth


In [6]:
val = evaluate(model, dataset, args, split='valid')
test = evaluate(model, dataset, args, split='test')

print(f'valid NDCG@{args.topk}: {val[0]:.4f}, HR@{args.topk}: {val[1]:.4f}')
print(f'test  NDCG@{args.topk}: {test[0]:.4f}, HR@{args.topk}: {test[1]:.4f}')

valid NDCG@10: 0.3614, HR@10: 0.6700
test  NDCG@10: 0.3208, HR@10: 0.6400


In [7]:
recommend_user = 1
recommendations = recommend_topk(model, recommend_user, dataset, args, topk=5)
recommendations

[(18, 0.2685363292694092),
 (4, 0.11875267326831818),
 (12, 0.094166599214077),
 (19, -0.04347328841686249),
 (5, -0.40692049264907837)]

In [8]:
# Optional: use a full-data checkpoint. The architecture args must match that checkpoint.
# args.interactions_path = str(PROJECT_ROOT / 'data' / 'processed' / 'bpi2012_complete_only' / 'sasrec_interactions.txt')
# args.checkpoint_path = str(PROJECT_ROOT / 'outputs' / '...' / 'sasrec_best.pth')
# args.hidden_units = 50
# args.num_blocks = 2
# args.num_heads = 1
# args.maxlen = 50